# Etapa 1: Aprendizado Simbólico (Árvore de decisão)
**Pergunta prática:** A escola possui infraestrutura digital básica?
**Domínio:** Educação (Censo Escolar / INEP - Amazonas)

A classe 1 indica internet para alunos e pelo menos um tipo de equipamento disponível para eles. O CSV filtrado já acompanha o projeto. Para recriá-lo a partir dos microdados, use separadamente etapa1_gerar_csv.py.

## 3. Cálculos Manuais de Entropia e Ganho de Informação

Nesta etapa, realizamos o cálculo manual da entropia e do ganho de informação para os três primeiros níveis da árvore de decisão, utilizando os microdados filtrados do Amazonas ($N = 5.562$ escolas).

### A. Entropia do Nó Raiz (Nível 0)

- **Total de exemplos ($N$):** $5.562$
- **Classe 0 (Inadequado):** $4.555$ escolas ($\approx 81,90\%$)
- **Classe 1 (Adequado):** $1.007$ escolas ($\approx 18,10\%$)
- **Entropia Inicial ($E(S)$):** $0,68237$

### B. Ganho de Informação na Raiz (Atributos Candidatos)

- **`IN_LABORATORIO_INFORMATICA`**: **0,19153** *(Selecionado por apresentar o maior ganho)*
- **`IN_BIBLIOTECA`**: $0,09893$
- **`TP_LOCALIZACAO`**: $0,09528$
- **`TP_DEPENDENCIA`**: $0,04378$
- **`IN_ENERGIA_GERADOR_FOSSIL`**: $0,00878$

### C. Níveis Seguintes

- **Nível 1 (Filhos de `IN_LABORATORIO_INFORMATICA`):**
    - Para $Lab = 0.0$, o atributo selecionado foi **`TP_LOCALIZACAO`** ($IG = 0,01869$).
    - Para $Lab = 1.0$, o atributo selecionado foi **`TP_DEPENDENCIA`** ($IG = 0,05249$).

Para cada nó, calculamos H = −Σ p log₂(p). O ganho é a entropia do nó menos a média das entropias dos filhos, ponderada pelo tamanho de cada filho. Na raiz, por exemplo, H = −(4555/5562) log₂(4555/5562) − (1007/5562) log₂(1007/5562) ≈ 0,6824. A tabela abaixo confere os três níveis com os dados.

Escrevendo h(p) para a entropia binária, podemos acompanhar um ramo inteiro. Cada fração dentro de h é a proporção da classe 1.
- Raiz (laboratório): h(1007/5562) − [4607 h(369/4607) + 955 h(638/955)]/5562 ≈ 0,1915.
- Sem laboratório (localização): h(369/4607) − [1107 h(181/1107) + 3500 h(188/3500)]/4607 ≈ 0,0187.
- Sem laboratório, localização urbana (dependência administrativa): h(181/1107) − [2 h(1/2) + 220 h(41/220) + 608 h(76/608) + 277 h(63/277)]/1107 ≈ 0,0108.

In [ ]:
from learn_decision_tree import ler_csv, treinar

alvo = "infra_digital_basica"
dados, atributos = ler_csv("dados_etapa1.csv", alvo, ["CO_ENTIDADE"])
historico = []
arvore = treinar(dados, atributos, alvo, 3, historico)
print("Nível | Escolas | Entropia | Atributo | Ganho")
for no in historico:
    atributo = no["escolhido"]
    print(no["profundidade"], no["n"], round(no["H"], 4), atributo, round(no["ganhos"][atributo], 4))

## Comparação com scikit-learn e validação em cinco partes

In [ ]:
import subprocess
import sys

def executar(script, *argumentos):
    resultado = subprocess.run([sys.executable, "-X", "utf8", script, *argumentos],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding="utf-8")
    print(resultado.stdout)
    resultado.check_returncode()

executar("comparar_sklearn.py", "dados_etapa1.csv")

In [ ]:
executar("validacao_cruzada.py", "dados_etapa1.csv")

Com profundidade 3, a árvore acerta em média 87,65% nos cinco testes. Prever sempre a classe mais frequente acerta 81,90%.

**Monotonicidade.** Imagine uma folha que prevê pela maioria. Com duas escolas da classe 0 e uma da classe 1, ela prevê 0. Se incluirmos mais duas escolas da classe 1 com as mesmas pistas e treinarmos de novo, a maioria passa a ser 1. A previsão para uma escola que já estava nessa folha muda.